In [1]:
!pip install ragas==0.1.9

In [2]:
!pip install \
langchain==0.2.14 \
langchain-core==0.2.32 \
langchain-community==0.2.12 \
langchain-groq==0.1.9

In [3]:
!pip install numpy==1.26.4
!pip install torch
!pip install sentence-transformers

All libraries installed successfully!

In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [5]:
key = os.environ.get("OPENAI_API_KEY", "NOT FOUND")
print(f"API key loaded: {key[:8]}... ({'OK' if key != 'NOT FOUND' else 'MISSING'})")

API key loaded: gsk_TjJh... (OK)


In [6]:
from google.colab import files

print("A file picker will appear below...")
print("Select any PDF from your computer")
uploaded = files.upload()

A file picker will appear below...
Select any PDF from your computer


Saving 2_Bamboo plant  its uses.pdf to 2_Bamboo plant  its uses (5).pdf


In [7]:
for filename in uploaded.keys():
    print(f"Uploaded: {filename} ({len(uploaded[filename])} bytes)")

Uploaded: 2_Bamboo plant  its uses (5).pdf (3910751 bytes)


In [8]:
pdf_filename = list(uploaded.keys())[0]
print(f"\nWe will use: {pdf_filename}")


We will use: 2_Bamboo plant  its uses (5).pdf


In [9]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_and_chunk(pdf_path):
    # Step 1: Load the PDF
    # PyPDFLoader reads the PDF and extracts text page by page
    print(f"Loading PDF: {pdf_path}")
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    print(f"Loaded {len(pages)} pages")

    # Step 2: Split into chunks
    # RecursiveCharacterTextSplitter tries to split on paragraphs first,
    # then sentences, then words — always keeping meaning intact
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,    # each chunk = max 512 characters
        chunk_overlap=64,  # 64 chars shared between neighbouring chunks
        separators=["\n\n", "\n", ". ", " ", ""]
        # tries to split on blank lines first, then single lines,
        # then sentences, then words — in that priority order
    )

    chunks = splitter.split_documents(pages)
    print(f"Created {len(chunks)} chunks from {len(pages)} pages")
    print(f"Average chunk size: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")

    # Preview first chunk so you can see what it looks like
    print(f"\nFirst chunk preview:")
    print("-" * 40)
    print(chunks[0].page_content[:300])
    print("-" * 40)

    return chunks

# Run it on your uploaded PDF
chunks = load_and_chunk(pdf_filename)
print(f"\nDone! {len(chunks)} chunks ready for embedding.")

Loading PDF: 2_Bamboo plant  its uses (5).pdf
Loaded 29 pages
Created 29 chunks from 29 pages
Average chunk size: 79 chars

First chunk preview:
----------------------------------------
TD 643: DESIGN OF BAMBOO 
STRUCTURES
By
Dr. Chaaruchandra Korde
Assistant Professor, Centre for Technology Alternative for Rural Areas
Indian Institute of Technology Bombay
BAMBOO CONSTRUCTION
L 2: Introduction to Bamboo
(13.01.2026)
----------------------------------------

Done! 29 chunks ready for embedding.


In [10]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from sentence_transformers import SentenceTransformer

def build_vector_store(chunks):
    print("Loading embedding model...")
    print("(This downloads ~90MB the first time — subsequent runs are instant)")

    # HuggingFaceEmbeddings loads a model that converts text to numbers
    # all-MiniLM-L6-v2 is fast, free, and good quality — perfect for learning
    # It runs on YOUR computer (or Colab's), not an external API — so it's free
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}  # use cpu (colab free tier)
    )

    print(f"\nEmbedding {len(chunks)} chunks...")
    print("(Converting each chunk to a list of 384 numbers)")

    # FAISS takes all chunks + their embeddings and builds a searchable index
    # Think of it like building the index at the back of a textbook —
    # you build it once so every future lookup is instant
    vectorstore = FAISS.from_documents(chunks, embeddings)

    # Save to Colab's file system
    vectorstore.save_local("/content/faiss_index")
    print("\nVector store saved to /content/faiss_index")

    # Quick test: search for something to verify it works
    test_results = vectorstore.similarity_search("main topic", k=2)
    print(f"\nTest search returned {len(test_results)} chunks — working!")
    print(f"Sample: {test_results[0].page_content[:150]}...")

    return vectorstore

# Build the vector store from our chunks
vectorstore = build_vector_store(chunks)
print("\nEmbeddings complete!")

Loading embedding model...
(This downloads ~90MB the first time — subsequent runs are instant)


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding 29 chunks...
(Converting each chunk to a list of 384 numbers)

Vector store saved to /content/faiss_index

Test search returned 2 chunks — working!
Sample: OUTLINE OF PRESENTATION 
• Bamboo Plant
• Uses of Bamboo 
(other than construction)
2...

Embeddings complete!


In [96]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def build_rag_chain(vectorstore):

    prompt_template = """
You are a precise document assistant.

Answer the question using ONLY the information provided in the context below.
Do NOT use any outside knowledge.

Your answer must be:
- correct (factually accurate based on context)
- complete (cover all relevant points from context)
- relevant (directly answer the question)

Structure your answer clearly using bullet points or short paragraphs.

If possible, quote exact phrases from the context to support your answer.

If the answer is not present in the context, respond exactly with:
"I cannot find this in the document."

Do not use bold text or emojis. Keep the answer in plain text.

Context:
{context}

Question:
{question}

Answer:
"""

    prompt = PromptTemplate.from_template(prompt_template)

    # LLM
    llm = ChatGroq(
    groq_api_key=key,
    model_name="llama-3.1-8b-instant"  # 🔥 change here
)

    # Retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Convert docs → text
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # RAG pipeline
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain


# Build chain
rag_chain = build_rag_chain(vectorstore)
print("RAG chain ready! 🚀")

RAG chain ready! 🚀


In [97]:
def ask(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Step 1: Retrieve relevant documents manually
    docs = vectorstore.similarity_search(question, k=4)

    # Step 2: Generate answer using your RAG chain
    result = rag_chain.invoke(question)

    print(f"Answer:\n{result}")
    print("\n" + "-" * 50)
    print("Sources used (chunks model read):")

    # Step 3: Show sources
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "?")
        snippet = doc.page_content[:200].replace("\n", " ")
        print(f"\n  Source {i} (Page {page}):")
        print(f"  '{snippet}...'")

    return {
        "answer": result,
        "source_documents": docs
    }

In [98]:
questions = [
    "Summarise this document in 3 sentences",
    "What methodology or approach is described?",
    "What are the limitations mentioned?",
    "What are the main recommendations?",
    "What data or evidence is presented?"
]

for q in questions:
    print(f"Q: {q}")

    # Step 1: Retrieve relevant chunks
    docs = vectorstore.similarity_search(q, k=4)

    # Step 2: Get answer from RAG chain
    answer = rag_chain.invoke(q)

    print(f"A: {answer[:300]}...")
    print(f"Sources: {len(docs)} chunks used")

    print("-" * 60)

Q: Summarise this document in 3 sentences
A: This document is an outline of a presentation about bamboo, specifically highlighting its uses beyond construction. 

It briefly mentions harvesting and cutting techniques for bamboo, referencing YouTube videos for further information. 

The document also emphasizes the multiple uses of bamboo, stat...
Sources: 4 chunks used
------------------------------------------------------------
Q: What methodology or approach is described?
A: The methodology or approach is described in the context of a study. 

According to the context:

"• A study conducted to determine the mode of 
formation of hole in a natural material like 
bamboo (Nogata & Takahashi, 1995)"

This indicates that the study's approach is to determine the mode of forma...
Sources: 4 chunks used
------------------------------------------------------------
Q: What are the limitations mentioned?
A: The limitations mentioned in the context are related to traditional methods of using tim

In [103]:
eval_questions = [
    "Name different types of bamboo",
    "When does culm grows to full length?",
    "Tell no. of species found in india",
    "what is the composition of bamboo culm?"
]

eval_ground_truth = [
    "Monopodial, sympodial",
    "80 – 180 DAYS FOR FULL LENGTH",
    "136 species in India",
    "A BAMBOO CULM CONSISTS OF AROUND 25 % OF PARENCHYMA, 40 % FIBERS AND 8 % CONDUCTING TISSUES"
]

print(f"Evaluation set ready: {len(eval_questions)} questions")
print("Now collecting answers from your RAG system...")

import time

eval_answers = []
eval_contexts = []

for q in eval_questions:
    docs = vectorstore.similarity_search(q, k=4)
    answer = rag_chain.invoke(q)

    eval_answers.append(answer)
    eval_contexts.append([d.page_content for d in docs])
    print(f"  Answered: {q[:50]}...")

print("\nAll answers collected. Ready to evaluate.")

Evaluation set ready: 4 questions
Now collecting answers from your RAG system...
  Answered: Name different types of bamboo...
  Answered: When does culm grows to full length?...
  Answered: Tell no. of species found in india...
  Answered: what is the composition of bamboo culm?...

All answers collected. Ready to evaluate.


In [104]:
min_len = min(len(eval_questions), len(eval_answers), len(eval_contexts), len(eval_ground_truth))

eval_questions = eval_questions[:min_len]
eval_answers = eval_answers[:min_len]
eval_contexts = eval_contexts[:min_len]
eval_ground_truth = eval_ground_truth[:min_len]

MANUAL VALIDATION

In [105]:
for i in range(len(eval_questions)):
    print(f"\nQ: {eval_questions[i]}")
    print(f"GT: {eval_ground_truth[i]}")
    print(f"ANS: {eval_answers[i]}")


Q: Name different types of bamboo
GT: Monopodial, sympodial
ANS: There are two types of bamboo mentioned in the context:

* Monopodial Bamboo
* Sympodial Bamboo

Additionally, the context mentions the variety of species of bamboo:

- 136 species are found in India
- 1200 species are found world over

Q: When does culm grows to full length?
GT: 80 – 180 DAYS FOR FULL LENGTH
ANS: According to the context, the growth of a bamboo culm takes between 80 - 180 days to reach its full length.

 Quote from the context: 
"• 80 – 180 DAYS FOR FULL LENGTH"

Q: Tell no. of species found in india
GT: 136 species in India
ANS: The number of species found in India is 136. 

The exact phrase from the context is:
- 136 species in India & 1200 species world over

Q: what is the composition of bamboo culm?
GT: A BAMBOO CULM CONSISTS OF AROUND 25 % OF PARENCHYMA, 40 % FIBERS AND 8 % CONDUCTING TISSUES
ANS: The composition of a bamboo culm is as follows:

* Around 25% PARENCHYMA
* Around 40% FIBERS
* Around

CHECKED 4 OUT OF 4 ANSWERS MATCHING
(MANUAL)

LLM AS EVALUATOR

In [106]:
def evaluate_answer(question, answer, ground_truth):
    prompt = f"""
You are an evaluator.

Question: {question}
Ground Truth: {ground_truth}
Model Answer: {answer}

Score from 0 to 10 based on:
- correctness
- completeness
- relevance

overall final score of the output to rate the model overall
"""

    return llm.invoke(prompt)

In [107]:
for i in range(len(eval_questions)):
    result = evaluate_answer(
        eval_questions[i],
        eval_answers[i],
        eval_ground_truth[i]
    )
    print(result)

content="Score Breakdown:\n\n1. **Correctness**: 9/10 (The model correctly identifies the two main types of bamboo: Monopodial and Sympodial. It also provides information about the number of species found in India and globally, which is accurate.)\n\n2. **Completeness**: 8/10 (While the model provides the two main types of bamboo, it does not go into further details or subcategories within these types. Additionally, it doesn't mention any specific characteristics or examples of these types.)\n\n3. **Relevance**: 9/10 (The model directly answers the question by providing the types of bamboo and some relevant statistics about bamboo species. However, it could be more relevant if it provided more context or details about these types.)\n\n**Overall Final Score**: 8.3/10\n\nThe model demonstrates a good understanding of the topic by providing accurate information and relevant statistics, but it could improve by providing more details, subcategories, or examples to make the answer more compr

AVERAGE = 8.3 + 10 + 7.7 + 8/4 = 8.5 out of 10

COSINE SIMILARITY VALIDATION

In [108]:
from sklearn.metrics.pairwise import cosine_similarity

def similarity_score(ans, gt):
    emb1 = embeddings.embed_query(ans)
    emb2 = embeddings.embed_query(gt)
    return cosine_similarity([emb1], [emb2])[0][0]

In [109]:
average = 0
for i in range(len(eval_questions)):
    average = average + similarity_score(eval_answers[i], eval_ground_truth[i])
    print("score for ans no.",i+1)
    print(similarity_score(eval_answers[i], eval_ground_truth[i]))
print("overall average cosine similarity score =",average/len(eval_questions))

score for ans no. 1
0.22872498938139535
score for ans no. 2
0.5652986371581415
score for ans no. 3
0.821611014152609
score for ans no. 4
0.9450038386777142
overall average cosine similarity score = 0.640159619842465


# FINAL VALIDATION SCORES:
MANUAL- 4 OUT OF 4 ANSWERS MATCHING

LLM -  8.5 out of 10

overall average cosine similarity score = 0.640159619842465

UI CODE
USING GRADIO

In [115]:
import gradio as gr

# We need to rebuild these inside functions because Gradio
# needs to handle new uploads each time
def process_pdf(pdf_file):
    """Called when user uploads a PDF"""
    global rag_chain, vectorstore

    if pdf_file is None:
        return "Please upload a PDF first."

    try:
        # Load and chunk the new PDF
        chunks = load_and_chunk(pdf_file.name)

        # Build new vector store for this PDF
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={"device": "cpu"}
        )
        vectorstore = FAISS.from_documents(chunks, embeddings)
        rag_chain = build_rag_chain(vectorstore)

        return f"Ready! Processed {len(chunks)} chunks from your PDF. Ask away!"
    except Exception as e:
        return f"Error processing PDF: {str(e)}"

def answer_question(question):
    if not question.strip():
        return "Please type a question.", ""

    try:
        answer = rag_chain.invoke(question)

        # manually get sources
        docs = vectorstore.similarity_search(question, k=4)

        sources = ""
        for i, doc in enumerate(docs, 1):
            page = doc.metadata.get("page", "?")
            snippet = doc.page_content[:200].replace("\n", " ")
            sources += f"Source {i} (Page {page+1}):\n{snippet}...\n\n"

        return answer, sources

    except Exception as e:
        return f"ERROR: {str(e)}", ""

# Build the interface
with gr.Blocks(title="RAG Document Intelligence") as demo:
    gr.Markdown("## Document Intelligence System")
    gr.Markdown("Upload any PDF, then ask questions about it.")

    with gr.Row():
        pdf_upload = gr.File(label="Upload PDF", file_types=[".pdf"])
        process_btn = gr.Button("Process Document", variant="primary")

    status = gr.Textbox(label="Status", interactive=False)
    process_btn.click(process_pdf, inputs=pdf_upload, outputs=status)

    question_input = gr.Textbox(
        label="Your question",
        placeholder="What is the main argument of this paper?",
        lines=2
    )
    ask_btn = gr.Button("Ask Question", variant="primary")

    with gr.Row():
        answer_output = gr.Textbox(label="Answer", lines=5)
        sources_output = gr.Textbox(label="Sources used", lines=5)

    ask_btn.click(
        answer_question,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )

# Launch — creates a public URL you can share!
demo.launch(share=True)  # share=True gives you a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe2d832731bc9ffa03.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
